# E3-06 신규 48곳 진단 (③ 태양·발전소·데이터)

2024-06 이후 발전을 시작한 48곳(시작 2024-06-03 ~ 2024-12-24)은 가상 평가연도 학습 기간(2022-06 ~ 2024-05)에 이력이 없다.
이 노트북은 기준 모델(E1-01x2)과 맑은날 기준값(E3-02bx2) 예측으로 신규·기존 발전소 오차를 비교한다.

필요한 것: `outputs/work/base.parquet`, `outputs/preds/E1-01x2.parquet`, `outputs/preds/E3-02bx2.parquet` (저장소 맨 위에서 실행).
지표는 발전소 단위 nMAE = Σ|예측 − 실적| / Σ실적(해 뜬 시각). 실제 벌금은 묶음 합계로 매기므로 방향을 보는 용도다.

## 요약 (2026-11-05 실행)

| | 신규 48곳 | 기존 205곳 |
|---|---|---|
| E1-01x2 nMAE | 24.70% | 22.37% |
| E1-01x2 편향(Σ오차/Σ실적) | +1.07% | +1.12% |
| E3-02bx2 nMAE | 24.51% | 23.02% |
| E3-02bx2 편향 | +1.37% | −1.08% |

- 신규 발전소 오차는 기존보다 상대적으로 약 10% 크다. 차이는 대부분 **시작 후 2 ~ 3번째 달**(nMAE 33 ~ 38%, 과대 예측 +5 ~ +9%)에서 나오고, 4번째 달부터는 기존 수준(17 ~ 24%)이다. 가동 초기의 출력 불안정으로 보인다.
- **실제 평가 기간(2025-06 ~)에는 48곳 모두 학습 이력이 5 ~ 12개월 있다**(마지막 시작 2024-12-24). 가상 평가연도의 "이력 0" 상황은 실제 제출에서는 생기지 않는다. 신규 발전소 문제는 가상 평가연도 점수에서 과장돼 보인다.
- 같은 관측지점(wb_station) 기존 발전소의 기준값을 빌리는 방법(E3-06x2, `add_clear_ref_oof_st`)은 E3-02bx2 대비 **+0.43%**로 개선이 없다 → 기각. 신규 48곳 중 같은 wb_station에 기존 발전소가 있는 곳은 24곳뿐이다.
- 참고: E3-02bx2는 발전소 단위 nMAE가 기존 발전소에서 오히려 커졌지만(22.37 → 23.02%) 벌금은 −2.35%다. 편향이 +1.12% → −1.08%로 바뀐 영향이 크다. 묶음 합계에서는 발전소 오차가 서로 상쇄되므로 편향이 중요하다(⑤ 묶음 보정과 관련).

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'analysis_lab':
    os.chdir('..')
sys.path.insert(0, 'src_tools')
import pandas as pd, numpy as np
from oibc_common import VALID_FROM

b = pd.read_parquet('outputs/work/base.parquet',
                    columns=['site_id', 'ts', 'energy', 'split', 'elev', 'start', 'wb_station', 'month'])
b['site_id'] = b.site_id.astype(str)
fit = b[(b.ts < VALID_FROM) & (b.ts >= VALID_FROM - pd.DateOffset(months=24)) & b.energy.notna()]
old = set(fit.site_id)                       # 학습 기간에 이력이 있는 발전소
v = b[(b.split == 'valid') & (b.elev > 0)].copy()
v['new'] = ~v.site_id.isin(old)
print('신규', v[v.new].site_id.nunique(), '곳, 시작', v[v.new].groupby('site_id').start.first().agg(['min', 'max']).tolist())

신규 48 곳, 시작 [Timestamp('2024-06-03 14:00:00+0900', tz='Asia/Seoul'), Timestamp('2024-12-24 15:00:00+0900', tz='Asia/Seoul')]


## 1. 신규·기존 nMAE와 편향

In [2]:
def nmae(g):
    return pd.Series({'nMAE_%': g.err.abs().sum() / g.energy.sum() * 100,
                      'bias_%': g.err.sum() / g.energy.sum() * 100, 'sites': g.site_id.nunique()})

merged = {}
for e in ['E1-01x2', 'E3-02bx2']:
    p = pd.read_parquet(f'outputs/preds/{e}.parquet'); p['site_id'] = p.site_id.astype(str)
    m = v.merge(p, on=['site_id', 'ts']).dropna(subset=['energy'])
    m['err'] = m.pred_kwh - m.energy
    merged[e] = m
    print(e); display(m.groupby('new')[['err', 'energy', 'site_id']].apply(nmae).round(2))

E1-01x2


,nMAE_%,bias_%,sites
new,,,
False,22.37,1.12,205.0
True,24.70,1.07,48.0


E3-02bx2


,nMAE_%,bias_%,sites
new,,,
False,23.02,-1.08,205.0
True,24.51,1.37,48.0


## 2. 신규 발전소: 시작 후 경과 개월별 nMAE (E1-01x2)

In [3]:
m = merged['E1-01x2']
n = m[m.new].copy()
n['months_since_start'] = ((n.ts - n.start).dt.days // 30).clip(upper=12)
n.groupby('months_since_start')[['err', 'energy', 'site_id']].apply(nmae).round(2)

,nMAE_%,bias_%,sites
months_since_start,,,
0,26.62,-6.34,48.0
1,32.75,4.58,48.0
2,38.19,8.80,48.0
3,25.00,-1.30,48.0
4,22.83,1.38,48.0
5,23.66,2.32,48.0
6,22.63,3.10,47.0
7,22.83,2.62,47.0
8,17.83,-0.13,47.0


## 3. 월별 nMAE: 신규 vs 기존 (E1-01x2)

In [4]:
pd.DataFrame({k: m[m.new == flag].groupby('month')[['err', 'energy', 'site_id']].apply(nmae)['nMAE_%']
              for k, flag in [('기존', False), ('신규', True)]}).round(1)

,기존,신규
month,,
1,22.4,23.5
2,16.7,31.6
3,19.1,25.6
4,17.1,17.5
5,21.9,21.8
6,20.2,25.2
7,32.8,31.7
8,21.2,22.1
9,29.1,28.6


## 4. 실제 평가 기간에서 신규 발전소 이력

In [5]:
t = pd.read_parquet('outputs/work/base_test.parquet', columns=['site_id', 'start']).drop_duplicates('site_id')
cut = pd.Timestamp('2025-06-01', tz='Asia/Seoul')
hist = ((cut - t.start).dt.days / 30).round(1)
print('2024-06 이후 시작:', (t.start >= pd.Timestamp('2024-06-01', tz='Asia/Seoul')).sum(), '곳')
print('실제 평가 기간 시작 시점 이력(개월):', hist[t.start >= pd.Timestamp('2024-06-01', tz='Asia/Seoul')].describe()[['min', 'max']].to_dict())

2024-06 이후 시작: 48 곳
실제 평가 기간 시작 시점 이력(개월): {'min': 5.3, 'max': 12.1}


## 5. 같은 관측지점 기존 발전소가 있는 신규 발전소

In [6]:
st = b.drop_duplicates('site_id')[['site_id', 'wb_station']].copy()
st['new'] = ~st.site_id.isin(old)
print(st[st.new].wb_station.isin(set(st[~st.new].wb_station)).sum(), '/', st.new.sum())
# E3-06x2 (관측지점 대체) 결과: python src_tools/evaluate.py outputs/preds/E3-06x2.parquet --ref outputs/preds/E3-02bx2.parquet
#   → 벌금 +0.43%, 11~15시 +0.51% (기각)

24

 / 48
